In [2]:
import pandas as pd
import mysql.connector
import os

dbd = "titanic"
folder = "D:\ZZ\Clean" 

conn = mysql.connector.connect(host="localhost", username="root", password="67777777")
cursor = conn.cursor()
cursor.execute(f"CREATE DATABASE IF NOT EXISTS `{dbd}`")
cursor.execute(f"USE `{dbd}`")
conn.close()
conn = mysql.connector.connect(host="localhost", username="root", password="67777777", database=dbd)
cursor = conn.cursor()

# Delete Schema
other_schema_to_delete = "titanics"
cursor.execute(f"DROP DATABASE IF EXISTS `{other_schema_to_delete}`")
print(f"Schema `{other_schema_to_delete}` deleted (if it existed).")

# Delete Tables
tables_to_delete = ["orderss"]
for table in tables_to_delete:
    cursor.execute(f"DROP TABLE IF EXISTS `{table}`")
    print(f"Table `{table}` deleted (if it existed).")

def get_sql_type(dtype):
    if pd.api.types.is_integer_dtype(dtype):
        return 'INT'
    elif pd.api.types.is_float_dtype(dtype):
        return 'FLOAT'
    elif pd.api.types.is_bool_dtype(dtype):
        return 'BOOLEAN'
    elif pd.api.types.is_datetime64_any_dtype(dtype):
        return 'DATETIME'
    else:
        return 'TEXT'

# Process all CSV files in the folder
for file_name in os.listdir(folder):
    if file_name.endswith(".csv"):
        table_name = os.path.splitext(file_name)[0]  # Use file name as table name
        table_name = table_name.replace(' ', '_').replace('-', '_').replace('.', '_')  # sanitize

        file_path = os.path.join(folder, file_name)
        df = pd.read_csv(file_path)

        # Convert date/time columns
        date_cols = [col for col in df.columns if any(keyword in col.lower() for keyword in ["date", "time", "dt", "timestamp"])]
        for col in date_cols:
            df[col] = pd.to_datetime(df[col], errors='coerce')

        df = df.where(pd.notnull(df), None)  # Replace NaN with None
        df.columns = [col.replace(' ', '_').replace('-', '_').replace('.', '_') for col in df.columns]

        # Create table
        columns = ', '.join([f'`{col}` {get_sql_type(df[col].dtype)}' for col in df.columns])
        create_table_query = f'CREATE TABLE IF NOT EXISTS `{table_name}` ({columns})'
        cursor.execute(create_table_query)

        # Insert data
        for _, row in df.iterrows():
            values = tuple(None if pd.isna(x) else x for x in row)
            sql = f"INSERT INTO `{table_name}` ({', '.join(['`' + col + '`' for col in df.columns])}) VALUES ({', '.join(['%s'] * len(row))})"
            cursor.execute(sql, values)

        conn.commit()
        print(f"Imported `{file_name}` into table `{table_name}` successfully.")

conn.close()
print("All CSV files have been imported.")

<>:6: SyntaxWarning: invalid escape sequence '\Z'
<>:6: SyntaxWarning: invalid escape sequence '\Z'
C:\Users\visha\AppData\Local\Temp\ipykernel_20256\3561750972.py:6: SyntaxWarning: invalid escape sequence '\Z'
  folder = "D:\ZZ\Clean"


Schema `titanics` deleted (if it existed).
Table `orderss` deleted (if it existed).
Imported `survive.csv` into table `survive` successfully.
Imported `titanic.csv` into table `titanic` successfully.
All CSV files have been imported.
